# NLP-Based Resume and Job Description Matching System

**Student:** Patel Aum Shirishkumar  
**Enrollment No:** 12302040601004  
**Department:** Computer Engineering, A. D. Patel Institute of Technology (CVM University)  
**Subject:** Natural Language Processing (Course Code: 202047809)  

---
### Abstract
This notebook demonstrates a domain-aware, multi-stage hybrid ranking architecture that combines:
1. **Dense Contextual Embeddings** (Sentence-BERT `all-MiniLM-L6-v2`) with section-weighted cosine similarity.
2. **Sparse Lexical Retrieval** (Sub-linear TF-IDF and Okapi BM25).
3. **Explicit Hard-Skill Verification** across 350+ technology competencies.

In [ ]:
# Step 1: Install prerequisites (if running on Google Colab)
!pip install sentence-transformers scikit-learn tabulate numpy

In [ ]:
# Step 2: Load Core Modules
import json
import os
import sys
from tabulate import tabulate

# Ensure local src is in path
sys.path.insert(0, os.path.abspath('.'))

from src.data_loader import DataLoader
from src.preprocessor import Preprocessor
from src.skill_extractor import SkillExtractor
from src.baseline_matcher import TFIDFMatcher, BM25Matcher
from src.semantic_matcher import SemanticMatcher
from src.hybrid_engine import HybridMatcher
from src.evaluate import Evaluator

loader = DataLoader()
taxonomy = loader.load_skills_taxonomy()
resumes = loader.load_resumes()
jds = loader.load_job_descriptions()

print(f"Loaded {len(resumes)} candidate resumes, {len(jds)} job descriptions, and {len(loader.get_all_skills_flat())} taxonomy skills.")

In [ ]:
# Step 3: Run Full Benchmark Suite across All 5 Retrieval Architectures
import run_experiments
run_experiments.main()

In [ ]:
# Step 4: Live Interactive Candidate Ranking for a Given Job Description
skill_extractor = SkillExtractor(taxonomy)
all_texts = [loader.get_resume_full_text(r) for r in resumes] + [loader.get_jd_full_text(j) for j in jds]

hybrid_engine = HybridMatcher(alpha=0.50, beta=0.20, gamma=0.30)
hybrid_engine.fit_corpus(all_texts)

# Select First Job Description: Senior NLP & ML Engineer
target_jd = jds[0]
ranked_candidates = hybrid_engine.rank_candidates(resumes, target_jd, skill_extractor, loader)

table_data = []
for rank, c in enumerate(ranked_candidates[:7], start=1):
    table_data.append([
        rank,
        c['candidate_name'],
        c['target_domain'],
        f"{c['hybrid_score']*100:.1f}%",
        f"{c['semantic_score']*100:.1f}%",
        f"{c['lexical_score']*100:.1f}%",
        f"{c['skill_score']*100:.1f}%",
        ", ".join(c['missing_required_skills'][:2]) if c['missing_required_skills'] else "None"
    ])

print(f"\n=== TOP RANKED CANDIDATES FOR: {target_jd['title']} ===")
print(tabulate(table_data, headers=["Rank", "Candidate", "Domain", "Hybrid", "Semantic", "Lexical", "Skill", "Missing Req"], tablefmt="grid"))